# 10 - Exploratory Data Analysis

This notebook analyzes the final analytical master tables and exports summary tables to `outputs/tables/`.

The analysis is organized around the project theme: **Cameroonian International Migration Trends (2015-2024): Destinations, Entry Reasons and Post-Arrival Trajectories Across Available Reference Years**.

The goal is to answer the three portfolio research questions while keeping indicators methodologically separated. The project does not aim to measure the causal effect of Covid-19; 2020 is treated only as a contextual reference point when relevant.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
PROJECT_ROOT = Path('..')

ANALYTICAL_PATH = PROJECT_ROOT / 'data' / 'processed' / 'analytical'
OUTPUTS_PATH = PROJECT_ROOT / 'outputs'
TABLES_PATH = OUTPUTS_PATH / 'tables'
FIGURES_PATH = OUTPUTS_PATH / 'figures'

TABLES_PATH.mkdir(parents=True, exist_ok=True)
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

period_order = ['pre_covid', 'covid', 'post_covid']


## Load Analytical Tables

The notebook uses the three master tables created in notebook 09.


In [ ]:
destinations = pd.read_csv(ANALYTICAL_PATH / 'destinations_master.csv')
entry_reasons = pd.read_csv(ANALYTICAL_PATH / 'entry_reasons_master.csv')
post_arrival = pd.read_csv(ANALYTICAL_PATH / 'post_arrival_master.csv')


In [ ]:
def prepare_eda_table(df):
    """Convert analytical columns to stable types before aggregation."""
    df = df.copy()

    if 'year' in df.columns:
        df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')

    if 'value' in df.columns:
        df['value'] = pd.to_numeric(df['value'], errors='coerce').fillna(0)

    text_columns = [
        'source',
        'dataset',
        'origin_country',
        'destination_country',
        'period',
        'measure_type',
        'reason',
        'sub_reason',
        'analysis_question',
    ]

    for col in text_columns:
        if col in df.columns:
            df[col] = df[col].astype('string').str.strip()

    return df


def export_csv(df, file_name):
    """Export a table to the portfolio output folder."""
    df.to_csv(TABLES_PATH / file_name, index=False)
    return df


In [ ]:
destinations = prepare_eda_table(destinations)
entry_reasons = prepare_eda_table(entry_reasons)
post_arrival = prepare_eda_table(post_arrival)

for name, df in {
    'destinations': destinations,
    'entry_reasons': entry_reasons,
    'post_arrival': post_arrival,
}.items():
    print()
    print(name.upper())
    print('Shape:', df.shape)
    print('Columns:', df.columns.tolist())
    if 'source' in df.columns:
        print('Sources:', sorted(df['source'].dropna().unique()))
    if 'measure_type' in df.columns:
        print('Measure types:', sorted(df['measure_type'].dropna().unique()))


## Q1 - Destination Countries

UN DESA stock data is the main source for Q1 because it provides the most consistent global view of Cameroonian migrant stocks by destination country. OECD indicators are exported separately as complementary evidence.


In [ ]:
undesa_destinations = destinations[
    (destinations['source'].str.lower() == 'undesa')
    & (destinations['measure_type'].str.lower() == 'stock')
].copy()

top_destinations_by_year = {}
for year in [2015, 2020, 2024]:
    top_table = (
        undesa_destinations[undesa_destinations['year'] == year]
        .groupby('destination_country', as_index=False)['value']
        .sum()
        .sort_values('value', ascending=False)
        .head(10)
    )
    top_destinations_by_year[year] = export_csv(top_table, f'top_10_destinations_{year}.csv')

top_destinations_by_year[2024]


In [ ]:
destination_evolution = (
    undesa_destinations[undesa_destinations['year'].isin([2015, 2020, 2024])]
    .pivot_table(
        index='destination_country',
        columns='year',
        values='value',
        aggfunc='sum',
    )
    .reset_index()
)

for year in [2015, 2020, 2024]:
    if year not in destination_evolution.columns:
        destination_evolution[year] = pd.NA

destination_evolution['change_2015_2024'] = destination_evolution[2024] - destination_evolution[2015]
destination_evolution['pct_change_2015_2024'] = (
    destination_evolution['change_2015_2024'] / destination_evolution[2015] * 100
)

top_increases = export_csv(
    destination_evolution.dropna(subset=[2015, 2024])
    .sort_values('change_2015_2024', ascending=False)
    .head(10),
    'top_destination_increases_2015_2024.csv',
)

top_decreases = export_csv(
    destination_evolution.dropna(subset=[2015, 2024])
    .sort_values('change_2015_2024', ascending=True)
    .head(10),
    'top_destination_decreases_2015_2024.csv',
)

top_increases


In [ ]:
oecd_destination_summary = (
    destinations[destinations['source'].str.lower() == 'oecd']
    .groupby(['measure_type', 'year'], as_index=False)['value']
    .sum()
    .sort_values(['measure_type', 'year'])
)

export_csv(oecd_destination_summary, 'oecd_destination_indicators_summary.csv')


## Q2 - Observable Reasons for Entry

Eurostat first permits are the main source for Q2. OECD worker inflow, seasonal worker inflow and asylum inflow are exported as complementary indicators only.


In [ ]:
eurostat_entry = entry_reasons[
    (entry_reasons['source'].str.lower() == 'eurostat')
    & (entry_reasons['measure_type'].str.lower() == 'permit')
].copy()

oecd_entry = entry_reasons[entry_reasons['source'].str.lower() == 'oecd'].copy()

reason_totals = export_csv(
    eurostat_entry.groupby('reason', as_index=False)['value'].sum().sort_values('value', ascending=False),
    'entry_reasons_total_by_reason.csv',
)

reason_yearly = export_csv(
    eurostat_entry.groupby(['year', 'reason'], as_index=False)['value'].sum().sort_values(['year', 'reason']),
    'entry_reasons_yearly_evolution.csv',
)

reason_by_period = (
    eurostat_entry.groupby(['period', 'reason'], as_index=False)['value'].sum()
)
reason_by_period['period'] = pd.Categorical(reason_by_period['period'], categories=period_order, ordered=True)
reason_by_period = export_csv(
    reason_by_period.sort_values(['period', 'reason']),
    'entry_reasons_by_period.csv',
)

period_totals = reason_by_period.groupby('period', as_index=False, observed=False)['value'].sum().rename(columns={'value': 'period_total'})
reason_share_by_period = reason_by_period.merge(period_totals, on='period', how='left')
reason_share_by_period['share_pct'] = reason_share_by_period['value'] / reason_share_by_period['period_total'] * 100
reason_share_by_period = export_csv(reason_share_by_period, 'entry_reasons_share_by_period.csv')

dominant_entry_reason_by_period = export_csv(
    reason_share_by_period.sort_values(['period', 'share_pct'], ascending=[True, False])
    .groupby('period', observed=False)
    .head(1)
    .reset_index(drop=True),
    'dominant_entry_reason_by_period.csv',
)

reason_totals


In [ ]:
oecd_entry_summary = (
    oecd_entry.groupby(['measure_type', 'sub_reason', 'year'], as_index=False)['value']
    .sum()
    .sort_values(['measure_type', 'sub_reason', 'year'])
)

export_csv(oecd_entry_summary, 'oecd_entry_indicators_summary.csv')


## Q3 - Post-Arrival Trajectories

Eurostat is used as the main source for Q3. OECD is used as complementary evidence only and is not merged into Eurostat post-arrival totals.


In [ ]:
eurostat_post_arrival = post_arrival[
    post_arrival['source'].str.lower() == 'eurostat'
].copy()

oecd_post_arrival = post_arrival[
    post_arrival['source'].str.lower() == 'oecd'
].copy()

post_arrival.groupby(['source', 'measure_type']).size().reset_index(name='rows')


In [ ]:
post_arrival_source_measure_summary = (
    post_arrival.groupby(['source', 'measure_type', 'year'], as_index=False)['value']
    .sum()
    .sort_values(['source', 'measure_type', 'year'])
)

export_csv(post_arrival_source_measure_summary, 'post_arrival_source_measure_summary.csv')


In [ ]:
post_arrival_totals = export_csv(
    eurostat_post_arrival.groupby('measure_type', as_index=False)['value'].sum().sort_values('value', ascending=False),
    'post_arrival_total_by_measure_type.csv',
)

post_arrival_yearly = export_csv(
    eurostat_post_arrival.groupby(['year', 'measure_type'], as_index=False)['value'].sum().sort_values(['year', 'measure_type']),
    'post_arrival_yearly_evolution.csv',
)

post_arrival_by_period = eurostat_post_arrival.groupby(['period', 'measure_type'], as_index=False)['value'].sum()
post_arrival_by_period['period'] = pd.Categorical(post_arrival_by_period['period'], categories=period_order, ordered=True)
post_arrival_by_period = export_csv(
    post_arrival_by_period.sort_values(['period', 'measure_type']),
    'post_arrival_by_period.csv',
)

post_arrival_totals


In [ ]:
top_countries_by_measure = (
    eurostat_post_arrival.groupby(['measure_type', 'destination_country'], as_index=False)['value']
    .sum()
    .sort_values(['measure_type', 'value'], ascending=[True, False])
)

top_10_post_arrival_countries_by_measure = export_csv(
    top_countries_by_measure.groupby('measure_type').head(10).reset_index(drop=True),
    'top_10_post_arrival_countries_by_measure.csv',
)

top_10_post_arrival_countries_by_measure.head(15)


In [ ]:
long_term_residents = eurostat_post_arrival[
    eurostat_post_arrival['measure_type'].str.lower() == 'long_term_resident'
].copy()

long_term_yearly = export_csv(
    long_term_residents.groupby('year', as_index=False)['value'].sum().sort_values('year'),
    'long_term_residents_yearly.csv',
)

long_term_by_country = export_csv(
    long_term_residents.groupby('destination_country', as_index=False)['value'].sum().sort_values('value', ascending=False),
    'long_term_residents_by_country.csv',
)

long_term_yearly


In [ ]:
status_changes = eurostat_post_arrival[
    eurostat_post_arrival['measure_type'].str.lower() == 'status_change'
].copy()

status_changes_yearly = export_csv(
    status_changes.groupby('year', as_index=False)['value'].sum().sort_values('year'),
    'status_changes_yearly.csv',
)

status_changes_by_country = export_csv(
    status_changes.groupby('destination_country', as_index=False)['value'].sum().sort_values('value', ascending=False),
    'status_changes_by_country.csv',
)

status_changes_yearly


In [ ]:
citizenship_acquisition = eurostat_post_arrival[
    eurostat_post_arrival['measure_type'].str.lower() == 'citizenship_acquisition'
].copy()

citizenship_yearly = export_csv(
    citizenship_acquisition.groupby('year', as_index=False)['value'].sum().sort_values('year'),
    'citizenship_acquisition_yearly.csv',
)

citizenship_by_country = export_csv(
    citizenship_acquisition.groupby('destination_country', as_index=False)['value'].sum().sort_values('value', ascending=False),
    'citizenship_acquisition_by_country.csv',
)

citizenship_yearly


### OECD Complementary Post-Arrival Indicators

OECD post-arrival indicators are useful as supporting evidence, but they are not merged with Eurostat Q3 totals because source coverage and definitions may differ.


In [ ]:
oecd_post_arrival_indicators_summary = (
    oecd_post_arrival.groupby(['measure_type', 'year'], as_index=False)['value']
    .sum()
    .sort_values(['measure_type', 'year'])
)

export_csv(oecd_post_arrival_indicators_summary, 'oecd_post_arrival_indicators_summary.csv')


## Methodological Caution

- Stocks are not added to flows.
- Permits are not added to citizenship acquisitions.
- OECD indicators are complementary evidence.
- Each `measure_type` must be interpreted according to its own definition.
- Comparisons are made across available reference years, not as a causal Covid-19 impact study.
